## 🏗️ LeetCode 84: Largest Rectangle in Histogram
---

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> For any bar, the largest rectangle using its <b>full height</b> is decided by how far left and right you can stretch until you hit a bar shorter than the current one.
</div>

### 🛠️ The Mathematical Model

To calculate the area efficiently, we need the **width** defined by the boundaries:

$$\text{Area} = \text{height[i]} \times (\text{right\_boundary} - \text{left\_boundary} - 1)$$

---

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>The Stack</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"The Blocked Height"</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Heights stay in a stack until a <i>shorter</i> bar blocks them. Pop and calculate immediately.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Boundaries</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Mapping the Terrain"</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Precompute <code>left_smaller</code> and <code>right_smaller</code> arrays in two linear passes.</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> A nested loop approach results in $O(n^2)$. For LeetCode 84, you <b>must</b> use a Monotonic Stack or Precomputed Boundaries to achieve <b>$O(n)$</b> time complexity.
</div>

In [9]:
def largestRectangleArea(heights):
    # What stops a rectangle starting at heights[i] from expanding to the right?
    # What stops it from expanding to the left?
    max_area = 0
    #Calculate lest Both sides
    
    length = len (heights)
    widths: list[int] = [1] * length
    #prepare th Widths list
    for i, height in enumerate(heights):
        #traverse both right and left . As long as you meet a higher high increase width by 1 
        #Break if condition is not met
        left, right = i-1, i+1
        while left >= 0 and heights[left] >= height:
            widths[i] += 1
            left -= 1
        while right <= length -1 and heights[right] >= height:
            widths[i] += 1
            right+= 1        
    # Now get the max
    for i, height in enumerate(heights):
        max_area = max(max_area, height * widths[i])
    return max_area
            


# Test Cases
print(largestRectangleArea([2, 1, 5, 6, 2, 3])) # Expected: 10
print(largestRectangleArea([2, 4]))             # Expected: 4

10
4


## 🚀 The $O(N)$ Optimization: The Monotonic Stack
---

Instead of walking backwards for every single bar, we can use a **Stack** as a 'waiting room'.
As we iterate through the heights:
1. If a bar is taller than the one before it, it goes into the waiting room (Stack).
2. If we hit a bar that is **shorter** than the top of our stack, that short bar has officially **blocked** the rectangle of the taller bar from expanding any further right!

At that exact moment, we `pop()` the taller bar out of the stack, calculate its final area (because we know exactly where it started and where it was blocked), and update our `max_area`.

# 🏗️ Time Complexity Analysis: $O(N^2)$ vs. Amortized $O(N)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> A nested loop does not automatically mean $O(N^2)$. If every element is processed (pushed/popped) exactly once across the entire runtime, the complexity is <b>Amortized $O(N)$</b>.
</div>

---

## 📉 The $O(N^2)$ Trap: Worst-Case Scenarios

While early exits help on average, Big-O focuses on the **worst-case**. In a "Staircase" or "Flat" input (e.g., `[1, 2, 3, 4, 5]`), a naive check-left approach forces redundant comparisons.

$$\text{Total Operations} = \sum_{i=1}^{N-1} i = \frac{N(N-1)}{2} \rightarrow O(N^2)$$

| Input Type | Performance | Reason |
|------------|-------------|--------|
| **Jagged** | Fast | Early `break` triggers often |
| **Sorted** | **Slow** | Every element checks all previous elements |

---

## 🚀 The Stack Solution: Amortized $O(N)$

In a Monotonic Stack implementation, we use a `for` loop with a nested `while` loop. However, we analyze the **lifecycle** of an element rather than the loops in isolation.

### The Lifecycle Rule
1. **Push:** Each bar enters the stack exactly **once**.
2. **Pop:** Each bar leaves the stack exactly **once**.

Because an element cannot be popped if it isn't there, the `while` loop's total iterations across the *entire* program cannot exceed `N`.

---

## 🛠️ Mathematical Proof

The total work done is the sum of all pushes and all pops:

$$\text{Total Work} = \text{Work}_{\text{push}} + \text{Work}_{\text{pop}} = N + N = 2N$$

In Big-O notation, constants are dropped:
$$O(2N) \rightarrow O(N)$$

---

## ✅ Key Takeaway

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> The "Naive" approach repeats work on the same elements multiple times. The "Stack" approach ensures each element is "solved" once and discarded, leading to linear time complexity.
</div>

---

### 💻 Implementation Sketch (Python)
```python
def largestRectangleArea(heights):
    stack = [] # Stores indices
    max_area = 0
    
    for i, h in enumerate(heights):
        # The 'while' loop looks scary, but it only pops what was pushed!
        while stack and heights[stack[-1]] > h:
            height = heights[stack.pop()]
            width = i if not stack else i - stack[-1] - 1
            max_area = max(max_area, height * width)
        stack.append(i)
    return max_area

In [10]:
def largestRectangleArea_optimized(heights):
    max_area = 0
    stack = []  # Will store pairs of (starting_index, height)
    
    for index, height in enumerate(heights):
        start = index
        # If we hit a shorter bar, our waiting room (stack) rectangles are BLOCKED.
        # We must pop them and finalize their area right now.
        while stack and stack[-1][1] > height:
            popped_index, popped_height = stack.pop()
            # Calculate area: height * (current_index - started_index)
            max_area = max(max_area, popped_height * (index - popped_index))
            # The new shorter bar can effectively "reach back" to where the popped bar started
            start = popped_index
            
        # Add the current bar to the waiting room
        stack.append((start, height))
        
    # Cleanup: Any bars left in the stack were never blocked on the right!
    # That means they extend all the way to the end of the array.
    for index, height in stack:
        max_area = max(max_area, height * (len(heights) - index))
        
    return max_area

print("Stack Solution:", largestRectangleArea_optimized([2, 1, 5, 6, 2, 3])) # Expected: 10
print("Stack Solution:", largestRectangleArea_optimized([2, 4]))             # Expected: 4

Stack Solution: 10
Stack Solution: 4
